# Multimodal Emotion Recognition System

Shared emotion space for FER/FER+ image model + text emotion model:

- angry
- fear
- happy
- sad
- surprise


In [ ]:
!pip install -q numpy pandas tensorflow opencv-python matplotlib seaborn scikit-learn kaggle pillow

## Setup

Install Kaggle credentials, define shared labels, and create export folders.


In [ ]:
from google.colab import files
import json
import os
import pathlib
import pickle
import shutil
import tarfile
import urllib.request
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid', palette='deep')

SHARED_LABELS = ['angry', 'fear', 'happy', 'sad', 'surprise']
LABEL_TO_INDEX = {label: idx for idx, label in enumerate(SHARED_LABELS)}
INDEX_TO_LABEL = {idx: label for label, idx in LABEL_TO_INDEX.items()}

IMG_SIZE = (48, 48)
COLOR_MODE = 'grayscale'
RESCALE_FACTOR = 1.0 / 255.0
BATCH_SIZE = 32
VOCAB_SIZE = 20000
MAX_LEN = 150
OOV_TOKEN = '<OOV>'
ARTIFACTS_DIR = pathlib.Path('emotion_model_artifacts')
ZIP_PATH = pathlib.Path('emotion_model_artifacts.zip')

print('Upload kaggle.json when picker opens.')
uploaded = files.upload()
if not uploaded:
    raise ValueError('No kaggle.json uploaded.')

uploaded_name = next(iter(uploaded))
kaggle_dir = pathlib.Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / 'kaggle.json').write_bytes(uploaded[uploaded_name])
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

if ARTIFACTS_DIR.exists():
    shutil.rmtree(ARTIFACTS_DIR)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('Shared labels:', SHARED_LABELS)
print('Artifacts dir:', ARTIFACTS_DIR.resolve())

## Shared Helpers

Reusable plotting, evaluation, JSON-safe export, and packaging helpers.


In [ ]:
def to_python(value):
    if isinstance(value, dict):
        return {str(k): to_python(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_python(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def save_json(path, payload):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(to_python(payload), f, indent=2)


def plot_history(history, title):
    hist_df = pd.DataFrame(history.history)
    hist_df['epoch'] = np.arange(1, len(hist_df) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=hist_df, x='epoch', y='accuracy', ax=axes[0], label='train', linewidth=2)
    sns.lineplot(data=hist_df, x='epoch', y='val_accuracy', ax=axes[0], label='validation', linewidth=2)
    axes[0].set_title(f'{title} Accuracy')
    axes[0].set_ylim(0, 1.05)

    sns.lineplot(data=hist_df, x='epoch', y='loss', ax=axes[1], label='train', linewidth=2)
    sns.lineplot(data=hist_df, x='epoch', y='val_loss', ax=axes[1], label='validation', linewidth=2)
    axes[1].set_title(f'{title} Loss')

    for ax in axes:
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()
    return hist_df


def export_history(history, prefix):
    hist_df = pd.DataFrame(history.history)
    hist_df.to_csv(ARTIFACTS_DIR / f'{prefix}_training_history.csv', index=False)
    save_json(ARTIFACTS_DIR / f'{prefix}_training_history.json', history.history)
    print(f'Saved history for {prefix}.')


def evaluate_predictions(y_true, y_prob, class_names, prefix, title):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.argmax(y_prob, axis=1)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    per_class_p, per_class_r, per_class_f1, per_class_support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(class_names))), zero_division=0
    )

    summary_df = pd.DataFrame(
        [
            ('accuracy', accuracy_score(y_true, y_pred)),
            ('balanced_accuracy', balanced_accuracy_score(y_true, y_pred)),
            ('macro_precision', macro_p),
            ('macro_recall', macro_r),
            ('macro_f1', macro_f1),
            ('weighted_precision', weighted_p),
            ('weighted_recall', weighted_r),
            ('weighted_f1', weighted_f1),
        ],
        columns=['metric', 'value']
    )

    per_class_df = pd.DataFrame({
        'label': class_names,
        'precision': per_class_p,
        'recall': per_class_r,
        'f1': per_class_f1,
        'support': per_class_support,
    })

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    summary_df.to_csv(ARTIFACTS_DIR / f'{prefix}_summary_metrics.csv', index=False)
    save_json(ARTIFACTS_DIR / f'{prefix}_summary_metrics.json', summary_df.to_dict(orient='records'))

    per_class_df.to_csv(ARTIFACTS_DIR / f'{prefix}_per_class_metrics.csv', index=False)
    save_json(ARTIFACTS_DIR / f'{prefix}_per_class_metrics.json', per_class_df.to_dict(orient='records'))

    cm_df.to_csv(ARTIFACTS_DIR / f'{prefix}_confusion_matrix.csv')
    save_json(
        ARTIFACTS_DIR / f'{prefix}_confusion_matrix.json',
        {'labels': class_names, 'matrix': cm.tolist()}
    )

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{title} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / f'{prefix}_confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.show()

    display(summary_df.round(4))
    display(per_class_df.round(4))
    return {'summary': summary_df, 'per_class': per_class_df, 'confusion_matrix': cm_df}


def copy_required_file(src_name, dest_name=None):
    src = pathlib.Path(src_name)
    if not src.exists():
        raise FileNotFoundError(f'Missing required file: {src_name}')
    shutil.copy2(src, ARTIFACTS_DIR / (dest_name or src.name))


def build_zip_bundle():
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(ARTIFACTS_DIR.rglob('*')):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(ARTIFACTS_DIR.parent))
    print('Created zip:', ZIP_PATH)

## FER+ Dataset Prep

Keep only `angry`, `fear`, `happy`, `sad`, `surprise`. Drop `disgust` and `neutral`.


In [ ]:
competition = 'challenges-in-representation-learning-facial-expression-recognition-challenge'
!kaggle competitions download -c $competition

competition_zip = pathlib.Path(f'{competition}.zip')
if not competition_zip.exists():
    raise FileNotFoundError('FER2013 download failed. Accept Kaggle competition rules first.')

with zipfile.ZipFile(competition_zip, 'r') as zf:
    zf.extractall('fer2013_comp')

fer_csv = next(pathlib.Path('fer2013_comp').rglob('fer2013.csv'), None)
if fer_csv is None:
    tar_candidates = list(pathlib.Path('fer2013_comp').rglob('*.tar.gz'))
    if not tar_candidates:
        raise FileNotFoundError('Could not find fer2013.csv after extraction.')
    with tarfile.open(tar_candidates[0], 'r:gz') as tar:
        tar.extractall('fer2013_comp_extracted')
    fer_csv = next(pathlib.Path('fer2013_comp_extracted').rglob('fer2013.csv'), None)
if fer_csv is None:
    raise FileNotFoundError('fer2013.csv missing after extraction.')

ferplus_csv = pathlib.Path('fer2013new.csv')
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv',
    ferplus_csv,
)

fer_df = pd.read_csv(fer_csv)
ferplus_df = pd.read_csv(ferplus_csv)
if len(fer_df) != len(ferplus_df):
    raise ValueError('FER2013 and FER+ row counts do not match.')

split_map = {'Training': 'train', 'PublicTest': 'validation', 'PrivateTest': 'test'}
ferplus_to_shared = {
    4: 'angry',
    6: 'fear',
    1: 'happy',
    3: 'sad',
    2: 'surprise',
}
emotion_cols = [
    'neutral', 'happiness', 'surprise', 'sadness', 'anger',
    'disgust', 'fear', 'contempt', 'unknown', 'NF'
]

fer_root = pathlib.Path('ferplus_filtered')
if fer_root.exists():
    shutil.rmtree(fer_root)
for split in split_map.values():
    for label in SHARED_LABELS:
        (fer_root / split / label).mkdir(parents=True, exist_ok=True)


def majority_label(votes):
    votes = votes.astype(float).copy()
    votes[votes <= 1.0] = 0.0
    total = votes.sum()
    if total <= 0:
        return None
    idx = int(np.argmax(votes))
    if votes[idx] <= 0.5 * total:
        return None
    return ferplus_to_shared.get(idx)


kept = 0
skipped = 0
split_counts = {split: 0 for split in split_map.values()}
class_counts = {split: {label: 0 for label in SHARED_LABELS} for split in split_map.values()}

for idx, (fer_row, ferplus_row) in enumerate(zip(fer_df.itertuples(index=False), ferplus_df.itertuples(index=False))):
    split = split_map.get(ferplus_row.Usage)
    if split is None:
        skipped += 1
        continue
    votes = np.array([getattr(ferplus_row, col) for col in emotion_cols], dtype=float)
    final_label = majority_label(votes)
    if final_label is None:
        skipped += 1
        continue
    pixels = np.fromstring(fer_row.pixels, sep=' ', dtype=np.uint8)
    if pixels.size != IMG_SIZE[0] * IMG_SIZE[1]:
        skipped += 1
        continue
    image = Image.fromarray(pixels.reshape(IMG_SIZE), mode='L')
    image.save(fer_root / split / final_label / f'fer_{idx:07d}.png')
    kept += 1
    split_counts[split] += 1
    class_counts[split][final_label] += 1

print('FER+ filtered dataset ready.')
print('Kept:', kept, '| Skipped:', skipped)
print('Split counts:', split_counts)
display(pd.DataFrame(class_counts).T[SHARED_LABELS])

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=RESCALE_FACTOR,
    rotation_range=12,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.08,
    zoom_range=0.10,
    brightness_range=(0.85, 1.15),
    horizontal_flip=True,
    fill_mode='nearest'
)
eval_datagen = ImageDataGenerator(rescale=RESCALE_FACTOR)

train_generator = train_datagen.flow_from_directory(
    fer_root / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)
validation_generator = eval_datagen.flow_from_directory(
    fer_root / 'validation',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=False
)
test_generator = eval_datagen.flow_from_directory(
    fer_root / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode='categorical',
    shuffle=False
)

if list(train_generator.class_indices.keys()) != SHARED_LABELS:
    raise ValueError(f'FER class order mismatch: {train_generator.class_indices}')

fer_class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
fer_class_weight_dict = {int(i): float(w) for i, w in enumerate(np.minimum(fer_class_weights, 4.0))}

print('FER class indices:', train_generator.class_indices)
print('FER class weights:', fer_class_weight_dict)

In [ ]:
def build_fer_model(num_classes=len(SHARED_LABELS), l2=1e-4):
    reg = regularizers.l2(l2)

    def conv_block(x, filters, drop_rate):
        x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, 3, padding='same', kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(drop_rate)(x)
        return x

    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1), name='image_input')
    x = conv_block(inputs, 64, 0.20)
    x = conv_block(x, 128, 0.25)
    x = conv_block(x, 256, 0.30)
    x = conv_block(x, 384, 0.35)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.25)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='emotion_probs')(x)
    return keras.Model(inputs, outputs, name='fer_model')


fer_model = build_fer_model()
fer_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

fer_callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_fer_model.keras', monitor='val_loss', save_best_only=True, verbose=1),
]

fer_model.summary()

In [ ]:
fer_history = fer_model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=60,
    validation_data=validation_generator,
    validation_steps=len(validation_generator),
    class_weight=fer_class_weight_dict,
    callbacks=fer_callbacks,
)

fer_model.save('fer_model.keras')
plot_history(fer_history, 'FER Model')
export_history(fer_history, 'fer')

validation_generator.reset()
fer_val_prob = fer_model.predict(validation_generator, verbose=0)
fer_val_metrics = evaluate_predictions(
    validation_generator.classes,
    fer_val_prob,
    SHARED_LABELS,
    'fer_validation',
    'FER Validation'
)

test_generator.reset()
fer_test_prob = fer_model.predict(test_generator, verbose=0)
fer_test_metrics = evaluate_predictions(
    test_generator.classes,
    fer_test_prob,
    SHARED_LABELS,
    'fer_test',
    'FER Test'
)

## Text Dataset Prep

Drop `love`, rename labels into shared label space, train/evaluate on official train/val/test files.


In [ ]:
!kaggle datasets download -d praveengovi/emotions-dataset-for-nlp

with zipfile.ZipFile('emotions-dataset-for-nlp.zip', 'r') as zf:
    zf.extractall('emotion_data')

raw_train = pd.read_csv('emotion_data/train.txt', sep=';', names=['text', 'emotion'])
raw_val = pd.read_csv('emotion_data/val.txt', sep=';', names=['text', 'emotion'])
raw_test = pd.read_csv('emotion_data/test.txt', sep=';', names=['text', 'emotion'])

TEXT_LABEL_RENAME = {
    'anger': 'angry',
    'fear': 'fear',
    'joy': 'happy',
    'sadness': 'sad',
    'surprise': 'surprise',
}


def prepare_text_split(df, split_name):
    df = df.copy()
    df['text'] = df['text'].astype(str).str.lower().str.strip()
    df = df[df['emotion'] != 'love'].copy()
    df['shared_label'] = df['emotion'].map(TEXT_LABEL_RENAME)
    df = df[df['shared_label'].isin(SHARED_LABELS)].copy()
    df['label'] = df['shared_label'].map(LABEL_TO_INDEX).astype(int)
    df = df[['text', 'emotion', 'shared_label', 'label']].reset_index(drop=True)
    print(f'{split_name} size:', len(df))
    display(df['shared_label'].value_counts().reindex(SHARED_LABELS, fill_value=0).rename(split_name).to_frame().T)
    return df


train_df = prepare_text_split(raw_train, 'train')
val_df = prepare_text_split(raw_val, 'validation')
test_df = prepare_text_split(raw_test, 'test')

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(train_df['text'])

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(train_df['text']), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(val_df['text']), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(test_df['text']), maxlen=MAX_LEN, padding='post', truncating='post')

y_train = train_df['label'].to_numpy()
y_val = val_df['label'].to_numpy()
y_test = test_df['label'].to_numpy()

text_class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
text_class_weight_dict = {int(i): float(w) for i, w in enumerate(text_class_weights)}

print('Text class weights:', text_class_weight_dict)
print('Train/Val/Test shapes:', X_train_pad.shape, X_val_pad.shape, X_test_pad.shape)

In [ ]:
EMBED_DIM = 256
TEXT_REG = regularizers.l2(1e-4)


def build_text_model(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN, num_classes=len(SHARED_LABELS)):
    inputs = keras.Input(shape=(max_len,), name='text_input')
    x = layers.Embedding(vocab_size, embed_dim, mask_zero=False)(inputs)
    x = layers.SpatialDropout1D(0.2)(x)
    x = layers.Conv1D(128, 5, activation='relu', padding='same', kernel_regularizer=TEXT_REG)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same', kernel_regularizer=TEXT_REG)(x)
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=TEXT_REG)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=TEXT_REG)(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='emotion_probs')(x)
    return keras.Model(inputs, outputs, name='text_model')


text_model = build_text_model()
text_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

text_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_text_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]

text_model.summary()

In [ ]:
text_history = text_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val_pad, y_val),
    class_weight=text_class_weight_dict,
    callbacks=text_callbacks,
)

text_model.save('text_model.keras')
plot_history(text_history, 'Text Model')
export_history(text_history, 'text')

text_val_prob = text_model.predict(X_val_pad, verbose=0)
text_val_metrics = evaluate_predictions(
    y_val,
    text_val_prob,
    SHARED_LABELS,
    'text_validation',
    'Text Validation'
)

text_test_prob = text_model.predict(X_test_pad, verbose=0)
text_test_metrics = evaluate_predictions(
    y_test,
    text_test_prob,
    SHARED_LABELS,
    'text_test',
    'Text Test'
)

## Export Artifacts

Save models, tokenizer, shared label maps, configs, metrics, history, README, zip, and download bundle.


In [ ]:
copy_required_file('best_fer_model.keras')
copy_required_file('fer_model.keras')
copy_required_file('best_text_model.keras')
copy_required_file('text_model.keras')

with open(ARTIFACTS_DIR / 'text_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

label_maps = {
    'shared_labels': SHARED_LABELS,
    'label_to_index': LABEL_TO_INDEX,
    'index_to_label': INDEX_TO_LABEL,
    'fer_class_indices': train_generator.class_indices,
    'text_class_indices': LABEL_TO_INDEX,
    'text_original_to_shared': TEXT_LABEL_RENAME,
}
save_json(ARTIFACTS_DIR / 'label_maps.json', label_maps)

inference_config = {
    'shared_labels': SHARED_LABELS,
    'fer': {
        'best_model_file': 'best_fer_model.keras',
        'final_model_file': 'fer_model.keras',
        'image_size': list(IMG_SIZE),
        'color_mode': COLOR_MODE,
        'rescale_factor': RESCALE_FACTOR,
        'class_indices': train_generator.class_indices,
    },
    'text': {
        'best_model_file': 'best_text_model.keras',
        'final_model_file': 'text_model.keras',
        'max_length': MAX_LEN,
        'vocabulary_size': VOCAB_SIZE,
        'fitted_vocabulary_size': min(VOCAB_SIZE, len(tokenizer.word_index) + 1),
        'oov_token': tokenizer.oov_token,
        'lowercase': True,
        'strip_whitespace': True,
        'class_indices': LABEL_TO_INDEX,
    },
}
save_json(ARTIFACTS_DIR / 'inference_config.json', inference_config)

readme_text = f"""# Emotion Model Artifacts

Shared label order:

```python
{SHARED_LABELS}
```

## Files

- `best_fer_model.keras`: best validation FER checkpoint
- `fer_model.keras`: final FER model after training
- `best_text_model.keras`: best validation text checkpoint
- `text_model.keras`: final text model after training
- `text_tokenizer.pkl`: fitted Keras tokenizer
- `label_maps.json`: shared label order and mapping metadata
- `inference_config.json`: preprocessing settings for local deployment

## Local Load Example

```python
import json
import pickle
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open('label_maps.json', 'r', encoding='utf-8') as f:
    label_maps = json.load(f)

with open('inference_config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

with open('text_tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

fer_model = keras.models.load_model('best_fer_model.keras')
text_model = keras.models.load_model('best_text_model.keras')

sentence = 'i feel very happy today'
seq = tokenizer.texts_to_sequences([sentence.lower().strip()])
pad = pad_sequences(seq, maxlen=config['text']['max_length'], padding='post', truncating='post')
pred = text_model.predict(pad, verbose=0)[0]
label = label_maps['shared_labels'][int(pred.argmax())]
print(label)
```
"""
(ARTIFACTS_DIR / 'README.md').write_text(readme_text, encoding='utf-8')

required_files = [
    'best_fer_model.keras',
    'fer_model.keras',
    'best_text_model.keras',
    'text_model.keras',
    'text_tokenizer.pkl',
    'label_maps.json',
    'inference_config.json',
    'README.md',
    'fer_training_history.csv',
    'fer_training_history.json',
    'text_training_history.csv',
    'text_training_history.json',
    'fer_validation_summary_metrics.csv',
    'fer_validation_summary_metrics.json',
    'fer_validation_per_class_metrics.csv',
    'fer_validation_per_class_metrics.json',
    'fer_validation_confusion_matrix.csv',
    'fer_validation_confusion_matrix.png',
    'fer_test_summary_metrics.csv',
    'fer_test_summary_metrics.json',
    'fer_test_per_class_metrics.csv',
    'fer_test_per_class_metrics.json',
    'fer_test_confusion_matrix.csv',
    'fer_test_confusion_matrix.png',
    'text_validation_summary_metrics.csv',
    'text_validation_summary_metrics.json',
    'text_validation_per_class_metrics.csv',
    'text_validation_per_class_metrics.json',
    'text_validation_confusion_matrix.csv',
    'text_validation_confusion_matrix.png',
    'text_test_summary_metrics.csv',
    'text_test_summary_metrics.json',
    'text_test_per_class_metrics.csv',
    'text_test_per_class_metrics.json',
    'text_test_confusion_matrix.csv',
    'text_test_confusion_matrix.png',
]

missing_files = [name for name in required_files if not (ARTIFACTS_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing artifact files: {missing_files}')

print('Exported files:')
for path in sorted(ARTIFACTS_DIR.rglob('*')):
    if path.is_file():
        print('-', path.relative_to(ARTIFACTS_DIR.parent))

build_zip_bundle()
if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Zip file not created: {ZIP_PATH}')

print('All required files verified.')
files.download(str(ZIP_PATH))